In [1]:
import requests

def get_dhs_sub_agencies():
    # DHS Top-tier Agency Code is '070'
    url = "https://api.usaspending.gov/api/v2/agency/070/sub_agency/?fiscal_year=2026"

    response = requests.get(url)
    if response.status_code == 200:
        sub_agencies = response.json().get('results', [])
        print(f"{'Sub-Agency Name':<50} | {'Obligations'}")
        print("-" * 70)
        for sa in sub_agencies:
            name = sa.get('name')
            obs = sa.get('total_obligations', 0)
            print(f"{name:<50} | ${obs:,.2f}")
    else:
        print(f"Error: {response.status_code}")

get_dhs_sub_agencies()

Sub-Agency Name                                    | Obligations
----------------------------------------------------------------------
U.S. Customs and Border Protection                 | $19,216,739,873.18
U.S. Coast Guard                                   | $4,188,524,381.44
Federal Emergency Management Agency                | $4,143,003,557.82
U.S. Immigration and Customs Enforcement           | $3,799,096,976.56
Office of Procurement Operations                   | $1,905,170,032.01
Transportation Security Administration             | $417,746,362.66
U.S. Citizenship and Immigration Services          | $413,949,822.05
Federal Law Enforcement Training Center            | $260,191,028.70
U.S. Secret Service                                | $117,716,874.08
Office of the Inspector General                    | $9,181,413.66


In [2]:
import requests
import json
from collections import defaultdict

def get_dhs_fy25_fixed():
    search_url = "https://api.usaspending.gov/api/v2/search/spending_by_award/"

    # Prefix mapping for DHS Components based on Award ID
    # This acts as a backup when the API returns 'None'
    DHS_PREFIXES = {
        "HSTS": "TSA (Transportation Security Admin)",
        "HSFE": "FEMA (Federal Emergency Mgmt Agency)",
        "HSCG": "U.S. Coast Guard",
        "HSBP": "CBP (Customs & Border Protection)",
        "HSSC": "USCIS (Citizenship & Immigration)",
        "HSIG": "Office of Inspector General",
        "70Z":  "U.S. Coast Guard (Modern Prefix)",
        "70B":  "CBP (Modern Prefix)",
    }

    payload = {
        "filters": {
            "time_period": [{"start_date": "2024-10-01", "end_date": "2025-09-30"}],
            "agencies": [{"type": "awarding", "tier": "toptier", "name": "Department of Homeland Security"}],
            "naics_codes": {"require": ["518210", "541511", "541512", "513210"]},
            "award_type_codes": ["A", "B", "C", "D"]
        },
        "fields": [
            "Award ID",
            "Recipient Name",
            "Award Amount",
            "Awarding Agency",
            "Awarding Sub Tier Agency"
        ],
        "limit": 100
    }

    try:
        response = requests.post(search_url, json=payload)
        if response.status_code == 200:
            results = response.json().get('results', [])

            component_totals = defaultdict(float)
            component_counts = defaultdict(int)

            for award in results:
                # 1. Try to get name from API field
                agency = award.get('Awarding Sub Tier Agency')

                # 2. If API returned None, use the Award ID prefix logic
                if not agency or agency == "None":
                    award_id = award.get('Award ID', "")
                    # Match the first 4 characters of the ID
                    prefix = award_id[:4]
                    agency = DHS_PREFIXES.get(prefix, "DHS - Other/Headquarters")

                amount = award.get('Award Amount') or 0.0
                component_totals[agency] += float(amount)
                component_counts[agency] += 1

            # Print Summary
            print(f"\n{'DHS COMPONENT (RESOLVED)':<45} | {'AWARDS':<7} | {'TOTAL OBLIGATED'}")
            print("-" * 80)
            for agency, total in sorted(component_totals.items(), key=lambda x: x[1], reverse=True):
                print(f"{agency[:43]:<45} | {component_counts[agency]:<7} | ${total:,.2f}")

        else:
            print(f"Error: {response.status_code}")
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    get_dhs_fy25_fixed()


DHS COMPONENT (RESOLVED)                      | AWARDS  | TOTAL OBLIGATED
--------------------------------------------------------------------------------
CBP (Customs & Border Protection)             | 2       | $617,988,126.68
USCIS (Citizenship & Immigration)             | 8       | $280,746,569.28
DHS - Other/Headquarters                      | 47      | $211,936,149.53
TSA (Transportation Security Admin)           | 5       | $71,998,203.87
U.S. Coast Guard                              | 28      | $62,344,523.90
FEMA (Federal Emergency Mgmt Agency)          | 9       | $34,140,981.26
Office of Inspector General                   | 1       | $1,770,707.10


# Set Consistent Query Parameters

In [5]:
# Oct 2024 through Sept 2025
start = '2020-10-01'
end = '2026-04-20'

target_departments = [
    "Department of Homeland Security", "Department of Justice",
    "Department of Defense",
    "Department of State", "Department of the Treasury",
    "Department of Energy", "Department of Commerce",
    "Department of Health and Human Services", "Department of Agriculture",
    "Department of the Interior", "Department of Transportation",
    "Securities and Exchange Commission", "Commodity Futures Trading Commission"
]

award_type_codes = ["A", "B", "C", "D"]

naics_codes = ["518210", "541511", "541519", "541512", "513210", "511210", "541512", "334111"]

# Award Spending - Use to Get Description of Contract

In [9]:
import requests
import json
import time
from datetime import datetime
from dateutil.relativedelta import relativedelta

search_url = "https://api.usaspending.gov/api/v2/search/spending_by_award/"
all_award_results = []

# Define your total range
start_dt = datetime.strptime(start, "%Y-%m-%d")
end_dt = datetime.strptime(end, "%Y-%m-%d")

for dept in target_departments:
    print('\n', '=' * 25, f"DEPARTMENT: {dept}")
    
    # Initialize current chunk start
    current_chunk_start = start_dt
    
    while current_chunk_start < end_dt:
        # Create a 3-month chunk (Quarterly)
        current_chunk_end = current_chunk_start + relativedelta(months=3) - relativedelta(days=1)
        
        # Ensure we don't overshoot the final end date
        if current_chunk_end > end_dt:
            current_chunk_end = end_dt
            
        s_str = current_chunk_start.strftime("%Y-%m-%d")
        e_str = current_chunk_end.strftime("%Y-%m-%d")
        
        print(f"--- Fetching: {s_str} to {e_str} ---")
        page = 1

        while True:
            payload = {
                "filters": {
                    "time_period": [{"start_date": s_str, "end_date": e_str}],
                    "agencies": [{"type": "awarding", "tier": "toptier", "name": dept}],
                    "award_type_codes": award_type_codes,
                    "naics_codes": {"require": naics_codes},
                },
                "fields": ["Award ID", "generated_internal_id", "Recipient Name", 
                           "Award Amount", "Awarding Agency", "Awarding Sub Agency", 
                           "Start Date", "End Date", "Description"],
                "limit": 100,
                "page": page,
                "order": "desc",
                "sort": "Award Amount"
            }

            try:
                response = requests.post(search_url, json=payload, timeout=60)
                response.raise_for_status()
                data = response.json()

                results = data.get('results', [])
                if not results:
                    break

                all_award_results.extend(results)
                print(f"Page {page}: Added {len(results)} records. Total: {len(all_award_results)}")

                if len(results) < 100:
                    break

                page += 1
                time.sleep(0.5) # Politeness delay

            except Exception as e:
                print(f"Error on {s_str} page {page}: {e}")
                print("Retrying in 5 seconds...")
                time.sleep(5)
                # This will loop back and try the same page again
                continue 

        # Move to the next 3-month chunk
        current_chunk_start += relativedelta(months=3)

print(f"\nDone! Combined total: {len(all_award_results)} awards.")


 ========================= DEPARTMENT: Department of Homeland Security
--- Fetching: 2020-10-01 to 2020-12-31 ---
Page 1: Added 100 records. Total: 100
Page 2: Added 100 records. Total: 200
Page 3: Added 100 records. Total: 300
Page 4: Added 100 records. Total: 400
Page 5: Added 100 records. Total: 500
Page 6: Added 100 records. Total: 600
Page 7: Added 100 records. Total: 700
Page 8: Added 100 records. Total: 800
Page 9: Added 100 records. Total: 900
Page 10: Added 100 records. Total: 1000
Page 11: Added 100 records. Total: 1100
Page 12: Added 100 records. Total: 1200
Page 13: Added 100 records. Total: 1300
Page 14: Added 100 records. Total: 1400
Page 15: Added 100 records. Total: 1500
Page 16: Added 100 records. Total: 1600
Page 17: Added 100 records. Total: 1700
Page 18: Added 100 records. Total: 1800
Page 19: Added 100 records. Total: 1900
Page 20: Added 100 records. Total: 2000
Page 21: Added 100 records. Total: 2100
Page 22: Added 100 records. Total: 2200
Page 23: Added 100 reco

In [10]:
import pandas as pd
df_awards = pd.DataFrame(all_award_results)
df_awards.to_csv('df_all_award_results.csv')

In [12]:
dfa = pd.read_csv('df_all_award_results.csv')
dfa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1075860 entries, 0 to 1075859
Data columns (total 13 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1075860 non-null  int64  
 1   internal_id            1075860 non-null  int64  
 2   Award ID               1075860 non-null  object 
 3   generated_internal_id  1075860 non-null  object 
 4   Recipient Name         1075860 non-null  object 
 5   Award Amount           1075860 non-null  float64
 6   Awarding Agency        1075860 non-null  object 
 7   Awarding Sub Agency    1075860 non-null  object 
 8   Start Date             1075853 non-null  object 
 9   End Date               1075860 non-null  object 
 10  Description            1075209 non-null  object 
 11  awarding_agency_id     1075860 non-null  int64  
 12  agency_slug            1075860 non-null  object 
dtypes: float64(1), int64(3), object(9)
memory usage: 106.7+ MB


# Transaction Spending Gives Better Picture of Yearly Spend
- Award spending could include all spending since inception of contract
- Cell below takes about 8 minutes to run (~57240 transactions)

In [14]:
import requests
import time
from datetime import datetime
from dateutil.relativedelta import relativedelta
from requests.exceptions import RequestException

# Config
search_url = "https://api.usaspending.gov/api/v2/search/spending_by_transaction/"
start_dt = datetime.strptime(start, "%Y-%m-%d")
end_dt = datetime.strptime(end, "%Y-%m-%d")

all_tx_results = []

for dept in target_departments:
    print('\n', '=' * 25, f"DEPARTMENT: {dept}")
    
    current_chunk_start = start_dt
    
    while current_chunk_start < end_dt:
        # Define the 6-month window
        current_chunk_end = current_chunk_start + relativedelta(months=6) - relativedelta(days=1)
        if current_chunk_end > end_dt:
            current_chunk_end = end_dt
            
        s_str = current_chunk_start.strftime("%Y-%m-%d")
        e_str = current_chunk_end.strftime("%Y-%m-%d")
        
        print(f"\n--- Range: {s_str} to {e_str} ---")
        page = 1

        while True:
            payload = {
                "filters": {
                    "time_period": [{"start_date": s_str, "end_date": e_str}],
                    "agencies": [{"type": "awarding", "tier": "toptier", "name": dept}],
                    "award_type_codes": award_type_codes,
                    "naics_codes": {"require": naics_codes},
                },
                "fields": [
                    "Award ID", "Recipient Name", "Action Date", 
                    "Transaction Amount", "Awarding Agency", "PSC"
                ],
                "limit": 100,
                "page": page,
                "sort": "Transaction Amount",
                "order": "desc"
            }

            success = False
            for attempt in range(5):
                try:
                    # 'timeout' is key to prevent hanging on a dead connection
                    response = requests.post(search_url, json=payload, timeout=45)
                    response.raise_for_status()
                    data = response.json()
                    success = True
                    break 
                except (RequestException, Exception) as e:
                    wait = 2 ** attempt
                    print(f"      [Retry {attempt+1}] Error: {e}. Waiting {wait}s...")
                    time.sleep(wait)

            if not success:
                print(f"      CRITICAL: Failed to retrieve {s_str} Page {page}. Skipping chunk.")
                break

            results = data.get('results', [])
            if not results:
                break

            all_tx_results.extend(results)
            print(f"      Page {page}: +{len(results)} tx (Total: {len(all_tx_results)})")

            # Check if we've reached the end of this date chunk
            if len(results) < 100:
                break

            page += 1
            time.sleep(0.2) # Small cooldown for the API

        # Move to the next 3-month block
        current_chunk_start = current_chunk_end + relativedelta(days=1)

print(f"\nCompleted! Final count: {len(all_tx_results)}")


 ========================= DEPARTMENT: Department of Homeland Security

--- Range: 2020-10-01 to 2021-03-31 ---
      Page 1: +100 tx (Total: 100)
      Page 2: +100 tx (Total: 200)
      Page 3: +100 tx (Total: 300)
      Page 4: +100 tx (Total: 400)
      Page 5: +100 tx (Total: 500)
      Page 6: +100 tx (Total: 600)
      Page 7: +100 tx (Total: 700)
      Page 8: +100 tx (Total: 800)
      Page 9: +100 tx (Total: 900)
      Page 10: +100 tx (Total: 1000)
      Page 11: +100 tx (Total: 1100)
      Page 12: +100 tx (Total: 1200)
      Page 13: +100 tx (Total: 1300)
      Page 14: +100 tx (Total: 1400)
      Page 15: +100 tx (Total: 1500)
      Page 16: +100 tx (Total: 1600)
      Page 17: +100 tx (Total: 1700)
      Page 18: +100 tx (Total: 1800)
      Page 19: +100 tx (Total: 1900)
      Page 20: +100 tx (Total: 2000)
      Page 21: +100 tx (Total: 2100)
      Page 22: +100 tx (Total: 2200)
      Page 23: +100 tx (Total: 2300)
      Page 24: +100 tx (Total: 2400)
      Page 25: +1

In [15]:
# Create df Transactions DataFrame
df_transactions = pd.DataFrame(all_tx_results)
df_transactions.to_csv('df_transactions.csv')

In [16]:
# Merge to df_awards
df_awards_abbrev = df_awards[['generated_internal_id', 'Description', 'Award Amount', 'Start Date', 'End Date']].copy()
df_transactions_description = pd.merge(df_transactions, df_awards_abbrev, how = 'left', on = 'generated_internal_id')

In [19]:
df_awards_abbrev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1075860 entries, 0 to 1075859
Data columns (total 5 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   generated_internal_id  1075860 non-null  object 
 1   Description            1075419 non-null  object 
 2   Award Amount           1075860 non-null  float64
 3   Start Date             1075853 non-null  object 
 4   End Date               1075860 non-null  object 
dtypes: float64(1), object(4)
memory usage: 41.0+ MB


In [17]:
df_transactions_description.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4613206 entries, 0 to 4613205
Data columns (total 12 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Award ID               object 
 1   Recipient Name         object 
 2   Action Date            object 
 3   Transaction Amount     float64
 4   Awarding Agency        object 
 5   PSC                    object 
 6   generated_internal_id  object 
 7   internal_id            int64  
 8   Description            object 
 9   Award Amount           float64
 10  Start Date             object 
 11  End Date               object 
dtypes: float64(2), int64(1), object(9)
memory usage: 422.4+ MB


In [ ]:
import requests
import time

def get_award_metadata_description(generated_id):
    url = f"https://api.usaspending.gov/api/v2/awards/{generated_id}/"

    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()

            # Strategy 1: Try the main award description
            desc = data.get('description')

            # Strategy 2: Fallback to Product/Service description if Strategy 1 is null
            if not desc and 'latest_transaction_contract_data' in data:
                desc = data['latest_transaction_contract_data'].get('product_or_service_description')

            return desc if desc else "No Description Available"
        return None
    except Exception as e:
        print(f"Error fetching {generated_id}: {e}")
        return None

# Get unique IDs where Description is currently NaN/Null
null_mask = df_transactions_description['Description'].isna()
unique_ids_to_fetch = df_transactions_description.loc[null_mask, 'generated_internal_id'].unique()

id_to_desc = {}

print(f"Fetching descriptions for {len(unique_ids_to_fetch)} unique awards...")

for i, g_id in enumerate(unique_ids_to_fetch):
    id_to_desc[g_id] = get_award_metadata_description(g_id)

    if i % 10 == 0:
        print(f"Processed {i}/{len(unique_ids_to_fetch)}")

    time.sleep(0.2)

# Update ONLY the null values in the original DataFrame - prevents overwriting existing good data
df_transactions_description.loc[null_mask, 'Description'] = df_transactions_description['generated_internal_id'].map(id_to_desc)

print("Backfill complete.")

Fetching descriptions for 534 unique awards...
Processed 0/534
Processed 10/534
Processed 20/534
Processed 30/534
Processed 40/534
Processed 50/534
Processed 60/534
Processed 70/534
Processed 80/534
Processed 90/534
Processed 100/534
Processed 110/534
Processed 120/534
Processed 130/534
Processed 140/534
Processed 150/534
Processed 160/534
Error fetching CONT_AWD_692M1522F00023_6920_DTFACT15D00003_6920: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Error fetching CONT_AWD_692M1523F00118_6920_DTFACT15D00003_6920: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Processed 170/534
Error fetching CONT_AWD_692M1525P00086_6920_-NONE-_-NONE-: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Error fetching CONT_AWD_693JJ322F000006_6925_47QTCA21D00DK_4732: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Error fetching

# Write Pulled USA Spending Data to CSV

In [ ]:
path = 'df_all_transactions.csv'

df_transactions_description.to_csv(path, index=False)

# Read in Previous CSV as DataFrame

In [ ]:
import pandas as pd

df_transactions_description = pd.read_csv('/content/drive/MyDrive/df_all_transactions.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# for row in df[df['is_intelligence'] == True]['Description']:
#   print(row)

# for index, row in df[df['is_graph_hw']].iterrows():
#   print(row['Awarding Sub Agency'])
#   print(row['Funding Sub Agency'])
#   print(row['PSC'])
#   print(row['Description'])
#   print(25 * '=')

# for d in df[df['is_intelligence']]['Description']:
#   print(d)

# df[df['is_graph_hw']]['Transaction Amount'].sum()

In [ ]:

df['intelligence_transactions'] = df['is_intelligence'] * df['Transaction Amount']
df['graph_hw_transactions'] = df['is_graph_hw'] * df['Transaction Amount']
df['graph_sw_transactions'] = df['is_graph_sw'] * df['Transaction Amount']
df['intelligence_graph_hw_transactions'] = df['is_intelligence'] * df['is_graph_hw'] * df['Transaction Amount']
df['intelligence_graph_sw_transactions'] = df['is_intelligence'] * df['is_graph_sw'] * df['Transaction Amount']

cols = ['Awarding Agency', 'Transaction Amount', 'intelligence_transactions', 'graph_hw_transactions', 'graph_sw_transactions',
        'intelligence_graph_hw_transactions', 'intelligence_graph_sw_transactions']

# Create Group By
df_gb = df[cols].groupby(['Awarding Agency']).sum().reset_index()

df_gb['percent_intelligence'] = df_gb['intelligence_transactions'] / df_gb['Transaction Amount']
df_gb['percent_graph'] = (df_gb['graph_hw_transactions'] + df_gb['graph_sw_transactions']) / df_gb['Transaction Amount']
df_gb['percent_intelligence_graph'] = (df_gb['intelligence_graph_hw_transactions'] + df_gb['intelligence_graph_sw_transactions']) / df_gb['intelligence_transactions']

df_gb.sort_values(by='percent_intelligence_graph', ascending=False)

,Awarding Agency,Transaction Amount,intelligence_transactions,graph_hw_transactions,graph_sw_transactions,intelligence_graph_hw_transactions,intelligence_graph_sw_transactions,percent_intelligence,percent_graph,percent_intelligence_graph
3,Department of Defense,1.531767e+10,4.247297e+08,1162574.42,3.647789e+08,0.0,10321847.43,0.027728,0.023890,0.024302
1,Department of Agriculture,7.947580e+08,4.999930e+04,0.00,0.000000e+00,0.0,0.00,0.000063,0.000000,0.000000
4,Department of Energy,7.195495e+08,1.015802e+06,0.00,0.000000e+00,0.0,0.00,0.001412,0.000000,0.000000
5,Department of Health and Human Services,5.037569e+09,1.936218e+07,0.00,1.501810e+07,0.0,0.00,0.003844,0.002981,0.000000
6,Department of Homeland Security,3.596920e+09,2.320157e+08,151481.00,1.250000e+05,0.0,0.00,0.064504,0.000077,0.000000
7,Department of Justice,1.122369e+09,1.171036e+07,0.00,4.779657e+06,0.0,0.00,0.010434,0.004259,0.000000
8,Department of State,1.135833e+09,1.414970e+05,45856.00,2.284805e+07,0.0,0.00,0.000125,0.020156,0.000000
9,Department of Transportation,8.885941e+08,1.679530e+04,19580.00,2.282281e+07,0.0,0.00,0.000019,0.025706,0.000000
10,Department of the Interior,8.777178e+08,3.015810e+06,0.00,1.485649e+06,0.0,0.00,0.003436,0.001693,0.000000
11,Department of the Treasury,1.909424e+09,2.871452e+07,831305.00,4.715218e+07,0.0,0.00,0.015038,0.025130,0.000000
